For the homework we are working with the green taxi data from october instead and not limiting it to 1000 rows.

In [1]:
import pandas as pd 


In [ ]:
from models import ride_serializer, GreenTaxiRide, ride_from_green_row

In [3]:
url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet'

In [4]:
columns = ['lpep_pickup_datetime', 'lpep_dropoff_datetime',	'PULocationID','DOLocationID','passenger_count','trip_distance','tip_amount','total_amount']
df = pd.read_parquet(url, columns=columns)

In [5]:
df.head()

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1.0,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1.0,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1.0,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1.0,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1.0,4.07,6.82,34.12


In [6]:
df.shape 
# we will eventually send this events somewhere (example a data lake) for future analysis or processing 

(49416, 8)

In [7]:
df.dtypes

lpep_pickup_datetime     datetime64[us]
lpep_dropoff_datetime    datetime64[us]
PULocationID                      int32
DOLocationID                      int32
passenger_count                 float64
trip_distance                   float64
tip_amount                      float64
total_amount                    float64
dtype: object

First we need to see how an event looks like, note that we will most likely be getting this data in json format. 

In [8]:
row = df.loc[0] 
row 


lpep_pickup_datetime     2025-10-01 00:21:47
lpep_dropoff_datetime    2025-10-01 00:24:37
PULocationID                             247
DOLocationID                              69
passenger_count                          1.0
trip_distance                            0.7
tip_amount                               1.7
total_amount                            10.0
Name: 0, dtype: object

After understanding the event (one taxi ride) we create a data class to represent the event. The data class is going to be added to the models.py file. The homework requires the datatime columns to be converted to string, why? JSON natively doesn't know how to handle Python datetime or Timestamp objects. If I try to json.dumps() a raw datetime, it will throw a TypeError.

In [9]:
df['lpep_pickup_datetime'] = df['lpep_pickup_datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')
df['lpep_dropoff_datetime'] = df['lpep_dropoff_datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')

# Handling any NaN values in passenger_count so it doesn't break the 'int' type
df['passenger_count'] = df['passenger_count'].fillna(0).astype(int)

In [10]:
df.dtypes

lpep_pickup_datetime      object
lpep_dropoff_datetime     object
PULocationID               int32
DOLocationID               int32
passenger_count            int64
trip_distance            float64
tip_amount               float64
total_amount             float64
dtype: object

In [11]:
print(type(df['lpep_pickup_datetime'].iloc[0]))

<class 'str'>


In [12]:
#creating one event exemple using this class
greenride = GreenTaxiRide( #TaxiRide as in the class from the models
    lpep_pickup_datetime= '2025-10-01 00:14:03',
    lpep_dropoff_datetime= '2025-10-01 00:20:00',
    PULocationID= 1,
    DOLocationID= 2,
    passenger_count= 1,
    trip_distance= 15.66,
    tip_amount= 2.0,
    total_amount= 15.76 
)


Kafka producers and consumers work with bytes, so two steps: 
- convert the event (ride) to dictionary
- convert the dictionary to json string and encode it to bytes.

In [13]:
ride_serializer(greenride) #ride_serializer = json serializer

b'{"lpep_pickup_datetime": "2025-10-01 00:14:03", "lpep_dropoff_datetime": "2025-10-01 00:20:00", "PULocationID": 1, "DOLocationID": 2, "passenger_count": 1, "trip_distance": 15.66, "tip_amount": 2.0, "total_amount": 15.76}'

In [2]:
# creating the producer
from kafka import KafkaProducer

servers = ['localhost:9092'] #one of the ports from redpanda docker container
producer = KafkaProducer(
    bootstrap_servers=servers,
    value_serializer=ride_serializer #turns the data into a string
)



NameError: name 'ride_serializer' is not defined

In [1]:
#sending the event to kafka
topic_name = 'g_rides' #topic is a place where we send data, this is our stream. in this case the stream is called 'g_rides'

producer.send(topic_name, value=greenride) #this value will be turn into bytes that kafka can read 
producer.flush()


NameError: name 'producer' is not defined

In [ ]:
import time

t0 = time.time()

for _, row in df.iterrows():
    greenride = ride_from_green_row(row)
    producer.send(topic_name, value=greenride)
    print(f"Sent: {greenride}")
    time.sleep(0.01)

producer.flush()

t1 = time.time()
print(f'Data exchange took {(t1 - t0):.2f} seconds')

NameError: name 'ride_from_green_row' is not defined